In [1]:
import boto3
import numpy as np
import os
import pathlib
import random
import tensorflow as tf
import time

from keras.preprocessing.image import img_to_array, load_img
from sagemaker import get_execution_role
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.tensorflow import TensorFlow
from sklearn.model_selection import train_test_split

Using TensorFlow backend.


# Creating a model using Tensorflow

## Configure hyperparameters

In [2]:
# Number of output classes
num_classes = 2

# Batch size for training
mini_batch_size =  128

# Max steps for training
max_steps = 5000

# Learning rate
learning_rate = 0.01

## Create a unique job name 

In [3]:
job_name_prefix = 'breast-cancer-detection'
timestamp = time.strftime('-%Y-%m-%d-%H-%M-%S', time.gmtime())
job_name = job_name_prefix + timestamp

## Specify the input paths for the job

In [4]:
bucket = 'alexbc.datalake.us-east-2'
input_prefix = 'breast-cancer-histopathology/curated-data/tf-recordio'
input_train = 's3://{}/{}/train/'.format(bucket, input_prefix)
input_test = 's3://{}/{}/test/'.format(bucket, input_prefix)

## Specify the output path for the job

In [5]:
output_prefix = 'breast-cancer-histopathology/model-output/tensorflow'
output_path = 's3://{}/{}/'.format(bucket, output_prefix)

## Configure training instances

In [6]:
instance_count = 1
instance_type = 'ml.p2.xlarge'
volume_size_gb = 50

## Get the execution role

In [7]:
role = get_execution_role()

## Configure train timeout

In [8]:
train_timeout = 360000

## Specify the path to the training script

In [9]:
training_script_path = 'tensorflowScript.py'

## Create a sagemaker.TensorFlow estimator

In [10]:
estimator = TensorFlow(entry_point=training_script_path,
                       role=role,
                       train_instance_count=instance_count,
                       train_instance_type=instance_type,
                       train_volume_size=volume_size_gb,
                       train_max_run=train_timeout,
                       model_dir=output_path,
                       output_path=output_path,
                       framework_version='1.12.0',
                       py_version = 'py3',
                       hyperparameters = {
                           'num-classes': num_classes,
                           'mini-batch-size': mini_batch_size,
                           'max-steps': max_steps,
                           'learning-rate': learning_rate
                       })

In [11]:
estimator.fit({
    'train': input_train,
    'validation': input_test
}, job_name=job_name)

2020-02-01 15:30:02 Starting - Starting the training job...
2020-02-01 15:30:04 Starting - Launching requested ML instances...
2020-02-01 15:31:01 Starting - Preparing the instances for training............
2020-02-01 15:32:39 Downloading - Downloading input data......
2020-02-01 15:34:00 Training - Training image download completed. Training in progress...2020-02-01 15:34:04,730 sagemaker-containers INFO     Imported framework sagemaker_tensorflow_container.training
2020-02-01 15:34:05,203 sagemaker-containers INFO     Invoking user script

Training Env:

{
    "additional_framework_parameters": {},
    "channel_input_dirs": {
        "validation": "/opt/ml/input/data/validation",
        "train": "/opt/ml/input/data/train"
    },
    "current_host": "algo-1",
    "framework_module": "sagemaker_tensorflow_container.training:main",
    "hosts": [
        "algo-1"
    ],
    "hyperparameters": {
        "num-classes": 2,
        "learning-rate": 0.01,
        "max-steps": 5000,
        

UnexpectedStatusException: Error for Training job breast-cancer-detection-2020-02-01-15-29-20: Failed. Reason: AlgorithmError: ExecuteUserScriptError:
Command "/usr/bin/python tensorflowScript.py --learning-rate 0.01 --max-steps 5000 --mini-batch-size 128 --model_dir s3://alexbc.datalake.us-east-2/breast-cancer-histopathology/mmodel-output/tensorflow/ --num-classes 2"